# SemanticEmbed: OpenTelemetry Traces → 6D Structural Analysis

Extract service dependencies from OTel trace data, encode the topology, and find structural risks your dashboards can't see.

[GitHub](https://github.com/jmurray10/semanticembed-sdk)

In [ ]:
!pip install -q semanticembed
from semanticembed import encode, report
print("Ready.")

---
## The Problem

OpenTelemetry gives you distributed traces — every request, every span, every hop. But traces answer **"what happened on this request?"** They don't answer:

- Which service is on 90% of all paths? (structural SPOF)
- Which service fans out to 12 others? (amplification risk)
- Which service has zero lateral redundancy? (convergence sink)

SemanticEmbed takes the service-to-service edges from your traces and computes structural risk from the topology alone.

---
## Step 1: Extract Edges from Trace Data

OTel traces are trees of spans. Each span has a `service.name` and a `parent_span_id`. The edges are the parent→child service relationships.

Below we simulate trace data. In production, you'd query your trace backend (Jaeger, Tempo, Honeycomb, Datadog, etc.).

In [ ]:
# Simulated OTel trace data — what you'd get from Jaeger/Tempo/etc.
# Each span has: trace_id, span_id, parent_span_id, service_name
traces = [
    # Trace 1: Checkout flow
    {"trace_id": "t1", "spans": [
        {"span_id": "s1", "parent": None,  "service": "frontend"},
        {"span_id": "s2", "parent": "s1",  "service": "api-gateway"},
        {"span_id": "s3", "parent": "s2",  "service": "checkout-service"},
        {"span_id": "s4", "parent": "s3",  "service": "payment-service"},
        {"span_id": "s5", "parent": "s3",  "service": "inventory-service"},
        {"span_id": "s6", "parent": "s3",  "service": "shipping-service"},
        {"span_id": "s7", "parent": "s4",  "service": "payment-provider"},
    ]},
    # Trace 2: Browse flow
    {"trace_id": "t2", "spans": [
        {"span_id": "s10", "parent": None,   "service": "frontend"},
        {"span_id": "s11", "parent": "s10",  "service": "api-gateway"},
        {"span_id": "s12", "parent": "s11",  "service": "product-catalog"},
        {"span_id": "s13", "parent": "s11",  "service": "recommendation-service"},
        {"span_id": "s14", "parent": "s13",  "service": "product-catalog"},
        {"span_id": "s15", "parent": "s12",  "service": "cache-service"},
    ]},
    # Trace 3: Cart flow
    {"trace_id": "t3", "spans": [
        {"span_id": "s20", "parent": None,   "service": "frontend"},
        {"span_id": "s21", "parent": "s20",  "service": "api-gateway"},
        {"span_id": "s22", "parent": "s21",  "service": "cart-service"},
        {"span_id": "s23", "parent": "s22",  "service": "redis-cart"},
        {"span_id": "s24", "parent": "s21",  "service": "currency-service"},
    ]},
]

print(f"Loaded {len(traces)} traces with {sum(len(t['spans']) for t in traces)} total spans")

In [ ]:
def extract_edges(traces):
    """Extract unique service-to-service edges from OTel traces."""
    edges = set()
    for trace in traces:
        # Build span_id -> service lookup
        span_to_service = {}
        for span in trace["spans"]:
            span_to_service[span["span_id"]] = span["service"]
        
        # Extract parent -> child service edges
        for span in trace["spans"]:
            if span["parent"] and span["parent"] in span_to_service:
                parent_svc = span_to_service[span["parent"]]
                child_svc = span["service"]
                if parent_svc != child_svc:  # skip self-calls
                    edges.add((parent_svc, child_svc))
    
    return sorted(edges)

edges = extract_edges(traces)
print(f"Extracted {len(edges)} unique service edges:")
for src, tgt in edges:
    print(f"  {src} -> {tgt}")

---
## Step 2: Encode the Topology

One function call. The encoding sees only the edges — no latency, no error rates, no trace IDs.

In [ ]:
result = encode(edges)

print(f"Nodes: {result.graph_info['nodes']}")
print(f"Edges: {result.graph_info['edges']}")
print(f"Max depth: {result.graph_info['max_depth']}")
print(f"Encoding time: {result.encoding_time_ms:.0f}ms")
print()
print(result.table)

---
## Step 3: Find Structural Risks

In [ ]:
risk = report(result)
print(risk)

---
## Step 4: Compare What OTel Shows vs What 6D Reveals

Your OTel dashboard tells you:
- `checkout-service` had P99 = 450ms on the last trace
- `payment-service` returned 2 errors today
- `redis-cart` is responding in <1ms

6D structural analysis tells you:

In [ ]:
dim_names = ["depth", "independence", "hierarchy", "throughput", "criticality", "fanout"]

print("STRUCTURAL INSIGHTS (invisible to OTel)")
print("=" * 55)

for node in ["api-gateway", "checkout-service", "product-catalog", "redis-cart"]:
    v = result.vectors[node]
    dims = dict(zip(dim_names, v))
    print(f"\n{node}:")
    
    # Flag structural insights
    insights = []
    if dims["independence"] < 0.2:
        insights.append("LOW INDEPENDENCE -- no parallel alternative at this depth")
    if dims["fanout"] > 0.7:
        insights.append(f"HIGH FANOUT ({dims['fanout']:.2f}) -- failure amplifies downstream")
    if dims["criticality"] > 0.05:
        insights.append(f"HIGH CRITICALITY ({dims['criticality']:.3f}) -- on many end-to-end paths")
    if dims["throughput"] > 0.2:
        insights.append(f"HIGH THROUGHPUT ({dims['throughput']:.2f}) -- carries significant traffic share")
    if dims["fanout"] == 0 and dims["depth"] > 0.7:
        insights.append("TERMINAL SINK -- deepest node, no downstream, no fallback")
    
    if insights:
        for i in insights:
            print(f"  >> {i}")
    else:
        print("  (no elevated structural risk)")

---
## Production Integration

### Query Your Trace Backend

Replace the simulated traces with real data from your backend:

**Jaeger:**
```python
import requests

resp = requests.get("http://jaeger:16686/api/traces", params={
    "service": "frontend",
    "limit": 1000,
    "lookback": "24h",
})
traces = resp.json()["data"]
edges = extract_edges_jaeger(traces)  # adapt the extraction
result = encode(edges)
```

**Grafana Tempo:**
```python
# Use TraceQL to find traces
resp = requests.get("http://tempo:3200/api/search", params={
    "q": '{resource.service.name="frontend"}',
    "limit": 1000,
})
```

**OTLP Collector Export:**
```python
# If you export traces to a file or database
import json
with open("traces.json") as f:
    traces = json.load(f)
edges = extract_edges(traces)
result = encode(edges)
```

### CI/CD Gate

Run structural analysis on every deployment:

```yaml
# .github/workflows/structural-check.yml
- name: Extract topology from recent traces
  run: python scripts/extract_edges.py > topology.json

- name: Structural risk check
  run: |
    python -c \"
    from semanticembed import encode_file, report
    result = encode_file('topology.json')
    risk = report(result)
    critical = risk.by_severity('critical')
    if critical:
        print('BLOCKING: Critical structural risks detected')
        print(risk)
        exit(1)
    print('No critical structural risks.')
    \"
```

---
## Your Own Traces

Paste your trace data below, or load from a file.

In [ ]:
# Option 1: Paste edges directly
my_edges = [
    ("service-a", "service-b"),
    ("service-a", "service-c"),
    ("service-b", "service-d"),
]

my_result = encode(my_edges)
print(my_result.table)
print()
print(report(my_result))

---

**Next:** [05 - AI Agent Pipelines](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/05_ai_agent_pipelines.ipynb)

*Patent pending. Application #63/994,075.*